In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
%sql
CREATE WIDGET TEXT catalog_name DEFAULT 'smartdata_project';
CREATE WIDGET TEXT schema_source DEFAULT 'silver';
CREATE WIDGET TEXT schema_sink DEFAULT 'gold';
CREATE WIDGET TEXT source_persons DEFAULT 'silver_persons';
CREATE WIDGET TEXT sink_persons DEFAULT 'gold_persons';
CREATE WIDGET TEXT w_persons DEFAULT '1900-01-01';

In [0]:
catalog_name = dbutils.widgets.get('catalog_name')
schema_source = dbutils.widgets.get('schema_source')
schema_sink = dbutils.widgets.get('schema_sink')
source_persons = dbutils.widgets.get('source_persons')
sink_persons = dbutils.widgets.get('sink_persons')
w_persons = dbutils.widgets.get('w_persons')

In [0]:
source_persons_table = f'{catalog_name}.{schema_source}.{source_persons}'
df_persons = spark.sql(f"SELECT * FROM {source_persons_table} WHERE updated_at > '{w_persons}'")

In [0]:
df_gold_persons = (
    df_persons
    .withColumn('class_age', 
        F.when(F.col('age') >= 50, 'Seniors')
        .when(F.col('age') <= 30, 'Mid-aged Adults').otherwise('Young Adults')
    )
    .withColumn('rfm', 
        F.when(F.col('rfm_id') == 1, 'Low')
        .when(F.col('rfm_id') == 2, 'Medium Low')
        .when(F.col('rfm_id') == 3, 'Medium')
        .when(F.col('rfm_id') == 4, 'Medium High')
        .when(F.col('rfm_id') == 5, 'High')
        .when(F.col('rfm_id') == 6, 'Diamond').otherwise('Unknown')
    )
    .drop('rfm_id')
)

In [0]:
target_sink_table = f'{catalog_name}.{schema_sink}.{sink_persons}'
if spark.catalog.tableExists(target_sink_table):

    source_table = DeltaTable.forName(spark, target_sink_table)
    df_gold_persons = df_gold_persons.withColumn('_ingestion_timestamp', F.current_timestamp())

    (
        source_table.alias("target")
        .merge(
            df_gold_persons.alias("source"),
            f"target.customer_id = source.customer_id"
        )
        # Registro existe y cambió algo → actualizar
        .whenMatchedUpdateAll()
        # Registro nuevo → insertar
        .whenNotMatchedInsertAll()
        .execute()
    )

else:
    (
        df_gold_persons.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .clusterBy('customer_id','updated_at','city')  # Liquid Clustering
        .saveAsTable(target_sink_table)
    )

In [0]:
max_date = df_persons.agg(F.max('updated_at')).collect()[0][0]

In [0]:
dbutils.jobs.taskValues.set(key="persons_date", value=str(max_date))